In [1]:
import google.generativeai as genai
import os

# --- Configuration ---
API_KEY_FILE = "apikey.txt"
# MODEL_NAME = 'gemini-pro' # チャット機能を持つ基本的なモデル
MODEL_NAME = 'gemini-2.5-flash-preview-05-20' # より高度な文脈理解が期待できるモデル
# MODEL_NAME = 'gemini-1.5-flash-latest' # 速度とコストのバランスが良いモデル

def get_api_key(filepath=API_KEY_FILE):
    """指定されたファイルからAPIキーを読み込む"""
    try:
        with open(filepath, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"エラー: APIキーファイル '{filepath}' が見つかりません。")
        return None

def start_conversation_with_llm(api_key, initial_question):
    """
    LLMとの対話を開始し、最初の質問に対する応答とチャットオブジェクトを返す。
    """
    if not api_key:
        print("APIキーが設定されていません。")
        return None, None

    genai.configure(api_key=api_key)
    try:
        model = genai.GenerativeModel(MODEL_NAME)
        # 新しいチャットセッションを開始
        chat = model.start_chat(history=[]) # 空の履歴で開始
    except Exception as e:
        print(f"モデルまたはチャットの初期化中にエラーが発生しました: {e}")
        return None, None

    print(f"\nあなた (最初の質問):\n{initial_question.strip()}")

    try:
        response = chat.send_message(initial_question)
        print(f"\nLLMの応答:\n{response.text.strip()}")
        return chat, response.text # チャットオブジェクトと最初の応答テキストを返す
    except Exception as e:
        print(f"LLMへのメッセージ送信中にエラーが発生しました: {e}")
        return chat, None # エラーが発生しても、チャットオブジェクトは返す試み

def continue_conversation(chat_session, follow_up_question):
    """
    既存のチャットセッションで追加の質問をする。
    """
    if not chat_session:
        print("エラー: 有効なチャットセッションがありません。")
        return None

    print(f"\nあなた (追加の質問):\n{follow_up_question.strip()}")
    try:
        response = chat_session.send_message(follow_up_question)
        print(f"\nLLMの応答:\n{response.text.strip()}")
        return response.text
    except Exception as e:
        print(f"LLMへのメッセージ送信中にエラーが発生しました: {e}")
        return None

if __name__ == "__main__":
    api_key = get_api_key()

    if api_key:
        tsubame_question = """
つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えました。
東急大井町線の大井町方面の電車に乗り換えたとき、各駅停車に乗車すべきところ、
間違えて急行に乗車してしまったことに気付きました。
自由が丘の次の急行停車駅で降車し、反対方向の電車で一駅戻った駅が
つばめちゃんの目的地でした。目的地の駅の名前を答えてください。
"""
        # 最初の質問をして、チャットセッションを開始
        chat_session, initial_response = start_conversation_with_llm(api_key, tsubame_question)

        if chat_session:
            print("\n--------------------------------------------------")
            print("最初の質問への応答は上記の通りです。")
            print("この会話の文脈は保持されています。")
            print("続けてこの内容について質問できます。")
            print("例: 'その駅は何線が通っていますか？' や 'その駅の近くに有名な場所はありますか？'")
            print("プログラムを終了する場合は何も入力せずにEnterキーを押してください。")
            print("--------------------------------------------------")

            # 追加の質問を受け付けるループ
            while True:
                try:
                    additional_question = input("\n追加の質問を入力してください: ").strip()
                    if not additional_question: # 何も入力されなければ終了
                        print("対話を終了します。")
                        break
                    continue_conversation(chat_session, additional_question)
                except KeyboardInterrupt: # Ctrl+Cで終了
                    print("\n対話を中断しました。")
                    break
                except Exception as e:
                    print(f"予期せぬエラーが発生しました: {e}")
                    break
    else:
        print("APIキーがないため、プログラムを実行できません。")

c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



あなた (最初の質問):
つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えました。
東急大井町線の大井町方面の電車に乗り換えたとき、各駅停車に乗車すべきところ、
間違えて急行に乗車してしまったことに気付きました。
自由が丘の次の急行停車駅で降車し、反対方向の電車で一駅戻った駅が
つばめちゃんの目的地でした。目的地の駅の名前を答えてください。

LLMの応答:
つばめちゃんの行動を順に追っていきましょう。

1.  **自由が丘駅**で東急東横線から東急大井町線に乗り換えました。
2.  **大井町方面**の電車に乗車しました。
3.  誤って**急行**に乗車してしまいました。
4.  自由が丘の次の急行停車駅で降車しました。東急大井町線の自由が丘から大井町方面の急行停車駅は、**大岡山（おおおかやま）**です。
5.  大岡山駅で降車し、そこから**反対方向**（二子玉川・溝の口方面）の電車に乗り換えました。
6.  反対方向に**一駅戻った駅**が目的地です。大岡山駅の溝の口方面の一つ手前の駅は、**緑が丘（みどりがおか）**です。

したがって、目的地の駅は **緑が丘** です。

--------------------------------------------------
最初の質問への応答は上記の通りです。
この会話の文脈は保持されています。
続けてこの内容について質問できます。
例: 'その駅は何線が通っていますか？' や 'その駅の近くに有名な場所はありますか？'
プログラムを終了する場合は何も入力せずにEnterキーを押してください。
--------------------------------------------------
対話を終了します。
